In [56]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [57]:
#Read Dataset
org_df = pd.read_csv('D:/CIS545_Proj/dataset/Airbnb_Open_Data.csv')
org_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102599 entries, 0 to 102598
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102599 non-null  int64  
 1   NAME                            102349 non-null  object 
 2   host id                         102599 non-null  int64  
 3   host_identity_verified          102310 non-null  object 
 4   host name                       102193 non-null  object 
 5   neighbourhood group             102570 non-null  object 
 6   neighbourhood                   102583 non-null  object 
 7   lat                             102591 non-null  float64
 8   long                            102591 non-null  float64
 9   country                         102067 non-null  object 
 10  country code                    102468 non-null  object 
 11  instant_bookable                102494 non-null  object 
 12  cancellation_pol

C:\Users\asus\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3185: DtypeWarning: Columns (25) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


In [58]:
#Drop duplicates data
org_df.drop_duplicates(inplace=True)

#Convert Price to float type and fill null value of service fee as 0
org_df['price'] = org_df['price'].replace('[\$,]', '', regex=True).astype(float)
org_df['service fee'] = org_df['service fee'].replace('[\$,]', '', regex=True).astype(float)


#Remove availabile days less than 0 
org_df['availability 365'] = np.where(org_df['availability 365']<0, org_df['availability 365']*-1, org_df['availability 365'])
#Remove availabile days more than 365
org_df['availability 365'] = np.where(org_df['availability 365']>365, 365, org_df['availability 365'])

#Remove minimum nights less than 0 
org_df['minimum nights'] = np.where(org_df['minimum nights']<0, org_df['minimum nights']*-1, org_df['minimum nights'])

#Remove typo in neighbourhood
org_df.rename(columns = {'neighbourhood group':'neighbourhood_group'}, inplace = True)
org_df = org_df[org_df.neighbourhood_group != 'brookln']

#Drop unimportant feature
cl_drop = ['id','NAME','host id','host name','country','country code','last review','reviews per month', 'license', 'neighbourhood','house_rules']
org_df.drop(columns=cl_drop, inplace=True)

# Drop null values
final_df = org_df.dropna()

#Encode possible oridinal object type features with LabelEncoder
cl_en = ['host_identity_verified','instant_bookable','cancellation_policy']
encoder = LabelEncoder()
for column in cl_en:
    final_df[column] = encoder.fit_transform(final_df[column])
    

#Encode object type features with LabelEncoder
encoder = OneHotEncoder(sparse_output=False)
encoded_array = encoder.fit_transform(final_df[['neighbourhood_group','room type']])
encoded_df = pd.DataFrame(
    encoded_array, 
    columns=encoder.get_feature_names_out(['neighbourhood_group', 'room type']),
    index=final_df.index
)
final_df = pd.concat([final_df, encoded_df], axis=1)
final_df.drop(columns=['neighbourhood_group','room type'], inplace=True)


final_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 99583 entries, 0 to 102044
Data columns (total 22 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   host_identity_verified             99583 non-null  int32  
 1   lat                                99583 non-null  float64
 2   long                               99583 non-null  float64
 3   instant_bookable                   99583 non-null  int32  
 4   cancellation_policy                99583 non-null  int32  
 5   Construction year                  99583 non-null  float64
 6   price                              99583 non-null  float64
 7   service fee                        99583 non-null  float64
 8   minimum nights                     99583 non-null  float64
 9   number of reviews                  99583 non-null  float64
 10  review rate number                 99583 non-null  float64
 11  calculated host listings count     99583 non-null  fl

C:\Users\asus\AppData\Local\Temp\ipykernel_28048\3827221115.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df[column] = encoder.fit_transform(final_df[column])
C:\Users\asus\AppData\Local\Temp\ipykernel_28048\3827221115.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df[column] = encoder.fit_transform(final_df[column])
C:\Users\asus\AppData\Local\Temp\ipykernel_28048\3827221115.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using

In [59]:
final_df.head(5)

,host_identity_verified,lat,long,instant_bookable,cancellation_policy,Construction year,price,service fee,minimum nights,number of reviews,...,availability 365,neighbourhood_group_Bronx,neighbourhood_group_Brooklyn,neighbourhood_group_Manhattan,neighbourhood_group_Queens,neighbourhood_group_Staten Island,room type_Entire home/apt,room type_Hotel room,room type_Private room,room type_Shared room
0,0,40.64749,-73.97237,0,2,2020.0,966.0,193.0,10.0,9.0,...,286.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,40.75362,-73.98377,0,1,2007.0,142.0,28.0,30.0,45.0,...,228.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0,40.68514,-73.95976,1,1,2005.0,368.0,74.0,30.0,270.0,...,322.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,1,40.79851,-73.94399,0,1,2009.0,204.0,41.0,10.0,9.0,...,289.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
5,1,40.74767,-73.97500,1,0,2013.0,577.0,115.0,3.0,74.0,...,365.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
